# Market + News Agent Mistral 7B QLoRA Training

This notebook fine-tunes **Mistral 7B** for the Market + Volatility + News Sentiment Agent using your project file:

`data/gold/market_sentiment_finetune_data.jsonl`

Use this in Google Colab with **T4 GPU** first. A full run on about 519 examples usually takes around **1-2 hours** on T4 depending on Colab speed.

Output adapter:

`models/market_mistral_lora/`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Change this only if your Drive folder path is different.
PROJECT_DIR = Path('/content/drive/MyDrive/298A')

FINETUNE_SCRIPT = PROJECT_DIR / 'agents' / 'finetune_lora.py'
INFER_SCRIPT = PROJECT_DIR / 'agents' / 'infer_lora.py'
TRAIN_JSONL = PROJECT_DIR / 'data' / 'gold' / 'market_sentiment_finetune_data.jsonl'
OUTPUT_DIR = PROJECT_DIR / 'models' / 'market_mistral_lora'

print('PROJECT_DIR:', PROJECT_DIR)
print('FINETUNE_SCRIPT:', FINETUNE_SCRIPT, FINETUNE_SCRIPT.exists())
print('INFER_SCRIPT:', INFER_SCRIPT, INFER_SCRIPT.exists())
print('TRAIN_JSONL:', TRAIN_JSONL, TRAIN_JSONL.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)


## 2. Install Dependencies

This installs the QLoRA stack. Colab already includes CUDA PyTorch, so do not reinstall torch unless Colab asks you to.

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes sentencepiece protobuf safetensors huggingface_hub


## 3. Check GPU

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print('GPU memory GB:', round(total_gb, 2))


## 4. Hugging Face Login

Mistral models may require accepting terms on Hugging Face and logging in. If download fails with a gated/auth error, run this cell and paste your HF token.

In [ ]:
# Run only if needed.
# from huggingface_hub import login
# login()


## 5. Validate Market Training JSONL

In [ ]:
import json
from collections import Counter

rows = []
with open(TRAIN_JSONL, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

labels = Counter()
for item in rows:
    assistant = json.loads(item['messages'][-1]['content'])
    label = assistant.get('market_sentiment_risk_label') or assistant.get('risk_label') or 'UNKNOWN'
    labels[label] += 1

print('examples:', len(rows))
print('label counts:', dict(labels))
print('\nSample user:')
print(rows[0]['messages'][1]['content'])
print('\nSample assistant:')
print(rows[0]['messages'][2]['content'])


## 6. Configure Mistral 7B

Start with `mistralai/Mistral-7B-Instruct-v0.3`. If this model is gated for your account, accept the model terms on Hugging Face or switch to another Mistral-compatible instruct model.

In [ ]:
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'

# Fast smoke test settings. Use this first.
EPOCHS = 1
MAX_LENGTH = 768
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = '2e-4'

print('BASE_MODEL:', BASE_MODEL)
print('OUTPUT_DIR:', OUTPUT_DIR)


## 7. Train Mistral QLoRA Adapter

This uses 4-bit QLoRA automatically through `agents/finetune_lora.py`.

For the first test, keep `EPOCHS = 1`. If the output looks good, change to `EPOCHS = 2` for the final run.

In [ ]:
!python3 "{FINETUNE_SCRIPT}" \
  --train-jsonl "{TRAIN_JSONL}" \
  --base-model "{BASE_MODEL}" \
  --output-dir "{OUTPUT_DIR}" \
  --max-length {MAX_LENGTH} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --lr {LR}


## 8. Verify Adapter Files

In [ ]:
!ls -lh "{OUTPUT_DIR}"
!test -f "{OUTPUT_DIR}/adapter_config.json" && echo 'adapter_config.json found'
!find "{OUTPUT_DIR}" -maxdepth 2 -name 'trainer_state.json' -print


## 9. Strict JSON Inference Test

In [ ]:
import json, subprocess, textwrap

prompt = json.dumps({
    'ticker': 'AAPL',
    'task': 'Assess market volatility and financial news sentiment risk.',
    'features': {
        'annualized_volatility': 0.28,
        'beta': 1.12,
        'max_drawdown': -0.17,
        'sentiment_mean': 0.03,
        'article_count': 40,
    },
})

cmd = [
    'python3', str(INFER_SCRIPT),
    '--base-model', BASE_MODEL,
    '--adapter-dir', str(OUTPUT_DIR),
    '--prompt', prompt,
    '--max-new-tokens', '220',
]

result = subprocess.run(cmd, text=True, capture_output=True)
print('returncode:', result.returncode)
print('\nSTDOUT:\n', result.stdout)
print('\nSTDERR:\n', result.stderr[-2000:])


## 10. Save Training Loss CSV

This extracts loss values from the latest checkpoint so your frontend/report can visualize the Mistral training curve.

In [ ]:
import csv, json

states = sorted(OUTPUT_DIR.glob('checkpoint-*/trainer_state.json'))
if not states:
    states = sorted(OUTPUT_DIR.glob('trainer_state.json'))

if states:
    state_path = states[-1]
    state = json.loads(state_path.read_text())
    out_csv = OUTPUT_DIR / 'training_loss.csv'
    with out_csv.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['step', 'epoch', 'loss', 'learning_rate', 'grad_norm'])
        writer.writeheader()
        for item in state.get('log_history', []):
            if 'loss' in item:
                writer.writerow({
                    'step': item.get('step'),
                    'epoch': item.get('epoch'),
                    'loss': item.get('loss'),
                    'learning_rate': item.get('learning_rate'),
                    'grad_norm': item.get('grad_norm'),
                })
    print('Saved:', out_csv)
else:
    print('No trainer_state.json found yet.')


## 11. Zip Adapter For Download / Drive Backup

In [ ]:
ADAPTER_ZIP = OUTPUT_DIR.parent / 'market_mistral_lora.zip'
!cd "{OUTPUT_DIR.parent}" && zip -qr market_mistral_lora.zip market_mistral_lora
!ls -lh "{ADAPTER_ZIP}"
print('Saved:', ADAPTER_ZIP)


## Local App Wiring After Training

If you download this adapter locally, set these in `.env`:

```env
MARKET_LORA_ENABLED=1
MARKET_LORA_BASE_MODEL=mistralai/Mistral-7B-Instruct-v0.3
MARKET_LORA_ADAPTER_DIR=models/market_mistral_lora
```

For local Mac inference, Mistral 7B may be slow. For fastest use, expose this adapter from Colab the same way you did with the Qwen LoRA endpoint and point `MARKET_LORA_REMOTE_URL` to the Colab `/predict` URL.